In [2]:
import glob
import numpy as np
import pandas as pd
import pandasql as ps
import polars as pl
import math
import itertools 

from xgboost import XGBClassifier
XGBOOST_AVAILABLE = True

#matplotlib libraries
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors
import seaborn as sns

#date libraries
from dateutil import parser
from datetime import datetime, timedelta, date


#prophet library

#pandas options
pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

#matplotlib setting defaults
sns.set(font="Arial",
        rc={
 "axes.axisbelow": False,
 "axes.edgecolor": "lightgrey",
 "axes.facecolor": "None",
 "axes.grid": False,
 "axes.labelcolor": "dimgrey",
 "axes.spines.right": False,
 "axes.spines.top": False,
 "figure.facecolor": "white",
 "lines.solid_capstyle": "round",
 "patch.edgecolor": "w",
 "patch.force_edgecolor": True,
 "text.color": "dimgrey",
 "xtick.bottom": False,
 "xtick.color": "dimgrey",
 "xtick.direction": "out",
 "xtick.top": False,
 "ytick.color": "dimgrey",
 "ytick.direction": "out",
 "ytick.left": False,
 "ytick.right": False})

In [3]:

def missing_data(input_data):
    '''
    This function returns dataframe with information about the percentage of nulls in each column and the column data type.
    
    input: pandas df
    output: pandas df
    
    '''
    
    total = input_data.isnull().sum()
    percent = (input_data.isnull().sum()/input_data.isnull().count()*100)
    table = pd.concat([total, percent], axis = 1, keys = ['Total', 'Percent'])
    types = []
    for col in input_data.columns: 
        dtype = str(input_data[col].dtype)
        types.append(dtype)
    table["Types"] = types
    return(pd.DataFrame(table))

def mape(actual, pred): 
    '''
    Mean Absolute Percentage Error (MAPE) Function
    
    input: list/series for actual values and predicted values
    output: mape value 
    '''
    actual, pred = np.array(actual), np.array(pred)
    return np.mean(np.abs((actual - pred) / actual)) * 100

In [4]:
data = glob.glob("bts_raw_data/*.zip")

In [5]:
import io
import zipfile

frames = []
for zip_path in data:
    with zipfile.ZipFile(zip_path) as archive:
        csv_name = next(name for name in archive.namelist() if name.lower().endswith('.csv'))
        with archive.open(csv_name) as csv_file:
            frames.append(pl.read_csv(io.BytesIO(csv_file.read())))

air_data = pl.concat(frames, how="diagonal_relaxed")

In [12]:
print(list(air_data.columns))

['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Flights', 'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'FirstDepTime', 'TotalAddGTime'

In [17]:
print(air_data.head())

shape: (5, 110)
┌──────┬─────────┬───────┬────────────┬───┬──────────────────┬───────────────┬─────────────┬──────┐
│ Year ┆ Quarter ┆ Month ┆ DayofMonth ┆ … ┆ Div5LongestGTime ┆ Div5WheelsOff ┆ Div5TailNum ┆      │
│ ---  ┆ ---     ┆ ---   ┆ ---        ┆   ┆ ---              ┆ ---           ┆ ---         ┆ ---  │
│ i64  ┆ i64     ┆ i64   ┆ i64        ┆   ┆ str              ┆ str           ┆ str         ┆ str  │
╞══════╪═════════╪═══════╪════════════╪═══╪══════════════════╪═══════════════╪═════════════╪══════╡
│ 2025 ┆ 4       ┆ 10    ┆ 10         ┆ … ┆ null             ┆               ┆             ┆ null │
│ 2025 ┆ 4       ┆ 10    ┆ 11         ┆ … ┆ null             ┆               ┆             ┆ null │
│ 2025 ┆ 4       ┆ 10    ┆ 12         ┆ … ┆ null             ┆               ┆             ┆ null │
│ 2025 ┆ 4       ┆ 10    ┆ 13         ┆ … ┆ null             ┆               ┆             ┆ null │
│ 2025 ┆ 4       ┆ 10    ┆ 14         ┆ … ┆ null             ┆               ┆      